# EllipseNet Pretrained — ResNet Backbone

This notebook keeps the same ellipse-detection idea as the clean NumPy notebook, but replaces the custom CNN backbone with a pretrained ResNet-18 feature extractor from torchvision.

The model still uses a YOLO-style grid:

- input image: `448 x 448`
- grid: `7 x 7`
- predictors per cell: `B = 2`
- output per predictor: `[objectness, cx, cy, a, b, theta, class logits]`

This version uses the YOLOv1-style 448x448 input while preserving a 7x7 prediction grid through adaptive pooling. It is meant to reduce early overfitting and improve feature quality by starting from ImageNet-pretrained visual features.


## Section 0 — Environment Check and Imports


In [ ]:
# If this cell fails because torch/torchvision is missing, run this once in a notebook cell:
# %pip install torch torchvision numpy pillow matplotlib

import os, json, time, math, xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse as MplEllipse

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from torchvision.models import resnet18, ResNet18_Weights
    TORCHVISION_WEIGHTS = ResNet18_Weights.DEFAULT
except Exception:
    from torchvision.models import resnet18
    TORCHVISION_WEIGHTS = None

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE)
print("torch:", torch.__version__)

np.random.seed(42)
torch.manual_seed(42)
os.makedirs("figures", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("checkpoints_pretrained", exist_ok=True)


## Section 1 — Configuration


In [ ]:
# ── Dataset paths ──────────────────────────────────────────────────────────
VOC_SEARCH_ROOT = "../archive/VOC2012_train_val/VOC2012_train_val"

# ── YOLO-style ellipse output ──────────────────────────────────────────────
IMG_SIZE = 448
S = 7
B = 2
C = 20

# ── Run mode ───────────────────────────────────────────────────────────────
# Start with staged or overfit runs. Set FULL_DATASET_RUN=True for the real run.
OVERFIT_SANITY_RUN = False
FULL_DATASET_RUN = False
STAGED_SAMPLES = None if FULL_DATASET_RUN else 200

MAX_TRAIN = 20 if OVERFIT_SANITY_RUN else STAGED_SAMPLES
MAX_VAL = 20 if OVERFIT_SANITY_RUN else STAGED_SAMPLES
VAL_SPLIT = "train" if OVERFIT_SANITY_RUN else "val"
TRAIN_RANDOM_ROTATION = not OVERFIT_SANITY_RUN
ROTATION_RANGE_DEG = (-20.0, 20.0)
ROTATION_FILL = (128, 128, 128)

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE = 4
EPOCHS = 20 if FULL_DATASET_RUN else 8
FREEZE_BACKBONE_EPOCHS = 2
LR_HEAD = 1e-3
LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

# ── Loss weights ──────────────────────────────────────────────────────────
L_LOC = 5.0
L_OBJ = 1.0
L_NOOBJ = 0.5
L_CLS = 1.0
L_THETA = 1.0
OBJECTNESS_TARGET = "binary"  # better recall for this small detector

# ── Evaluation ────────────────────────────────────────────────────────────
DEFAULT_CONF_THRESH = 0.3
CONF_THRESHOLDS = [0.3, 0.5, 0.7]
EVAL_IOU_THRESH = 0.5
NMS_IOU_THRESH = 0.45

if OVERFIT_SANITY_RUN:
    RUN_NAME = "pretrained_overfit"
elif FULL_DATASET_RUN:
    RUN_NAME = "pretrained_full"
else:
    RUN_NAME = f"pretrained_stage{STAGED_SAMPLES}"
CHECKPOINT_DIR = f"checkpoints_{RUN_NAME}"
LOG_FILE = f"outputs/train_log_{RUN_NAME}.json"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

VOC_CLASSES = [
    "aeroplane","bicycle","bird","boat","bottle","bus","car","cat",
    "chair","cow","diningtable","dog","horse","motorbike","person",
    "pottedplant","sheep","sofa","train","tvmonitor",
]
CLASS_TO_IDX = {name: i for i, name in enumerate(VOC_CLASSES)}

# Initial anchors are replaced by dataset-estimated anchors after loading train_ds.
ANCHORS = np.array([[0.08, 0.06], [0.20, 0.15]], dtype=np.float32)

print(f"run={RUN_NAME} img={IMG_SIZE} grid={S}x{S} B={B} C={C}")
print(f"train max={MAX_TRAIN} val split={VAL_SPLIT} val max={MAX_VAL}")
print(f"checkpoint dir={CHECKPOINT_DIR}")


## Section 2 — VOC Dataset and Rotation Augmentation


In [ ]:
def find_voc_root(search_root):
    needed = ("JPEGImages", "Annotations", "ImageSets")
    if all(os.path.isdir(os.path.join(search_root, d)) for d in needed):
        return search_root
    for root, dirs, _ in os.walk(search_root):
        if all(d in dirs for d in needed):
            return root
    raise FileNotFoundError(f"VOC root not found under {search_root!r}")


def normalise_theta(theta):
    return float(theta % np.pi)


def rotate_normalized_point(cx, cy, angle_deg):
    angle = np.deg2rad(angle_deg)
    dx, dy = cx - 0.5, cy - 0.5
    c, s = np.cos(angle), np.sin(angle)
    return float(0.5 + c * dx - s * dy), float(0.5 + s * dx + c * dy)


def rotate_sample(img, targets, angle_deg, fill=ROTATION_FILL):
    rotated_img = img.rotate(-angle_deg, resample=Image.BILINEAR,
                             expand=False, fillcolor=fill)
    theta_delta = np.deg2rad(angle_deg)
    rotated_targets = []
    for target in targets:
        cx, cy = rotate_normalized_point(target["cx"], target["cy"], angle_deg)
        if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0):
            continue
        new_target = dict(target)
        new_target["cx"] = cx
        new_target["cy"] = cy
        new_target["theta"] = normalise_theta(target["theta"] + theta_delta)
        rotated_targets.append(new_target)
    return rotated_img, rotated_targets


def parse_voc_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    W, H = int(size.findtext("width")), int(size.findtext("height"))
    targets = []
    for obj in root.iter("object"):
        name = obj.findtext("name", "")
        if name not in CLASS_TO_IDX or int(obj.findtext("difficult", "0")):
            continue
        b = obj.find("bndbox")
        xmin = max(0.0, float(b.findtext("xmin")))
        ymin = max(0.0, float(b.findtext("ymin")))
        xmax = min(W, float(b.findtext("xmax")))
        ymax = min(H, float(b.findtext("ymax")))
        if xmax <= xmin or ymax <= ymin:
            continue
        targets.append({
            "class_id": CLASS_TO_IDX[name],
            "cx": (xmin + xmax) / 2 / W,
            "cy": (ymin + ymax) / 2 / H,
            "a": (xmax - xmin) / 2 / W,
            "b": (ymax - ymin) / 2 / H,
            "theta": 0.0,
        })
    return {"file": root.findtext("filename"), "targets": targets}


def load_split(voc_root, split, max_n=None):
    split_file = os.path.join(voc_root, "ImageSets", "Main", f"{split}.txt")
    ids = [line.strip().split()[0] for line in open(split_file) if line.strip()]
    ann_dir = os.path.join(voc_root, "Annotations")
    samples = []
    for img_id in ids:
        xml = os.path.join(ann_dir, f"{img_id}.xml")
        if not os.path.exists(xml):
            continue
        sample = parse_voc_xml(xml)
        if sample["targets"]:
            samples.append(sample)
        if max_n is not None and len(samples) >= max_n:
            break
    return samples


class VOCEllipseTorchDataset(Dataset):
    def __init__(self, voc_root, split, max_n=None, random_rotation=False,
                 rotation_range=ROTATION_RANGE_DEG):
        self.img_dir = os.path.join(voc_root, "JPEGImages")
        self.samples = load_split(voc_root, split, max_n)
        self.random_rotation = bool(random_rotation)
        self.rotation_range = rotation_range
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = (Image.open(os.path.join(self.img_dir, sample["file"]))
               .convert("RGB")
               .resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
        targets = [dict(t) for t in sample["targets"]]
        if self.random_rotation:
            angle = float(np.random.uniform(*self.rotation_range))
            img, targets = rotate_sample(img, targets, angle)
        arr = np.array(img, dtype=np.float32).transpose(2, 0, 1) / 255.0
        tensor = torch.from_numpy(arr)
        tensor = (tensor - self.mean) / self.std
        return tensor, targets


def collate_fn(batch):
    imgs = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return imgs, targets


def estimate_anchors_from_samples(samples, B):
    axes = []
    for sample in samples:
        for target in sample["targets"]:
            axes.append([target["a"], target["b"]])
    axes = np.array(axes, dtype=np.float32)
    if len(axes) == 0:
        return ANCHORS.copy()
    order = np.argsort(axes[:, 0] * axes[:, 1])
    qs = np.linspace(0.2, 0.8, B)
    anchors = np.array([axes[order[int(q * (len(order) - 1))]] for q in qs], dtype=np.float32)
    for _ in range(25):
        inter = np.minimum(axes[:, None, 0], anchors[None, :, 0]) * np.minimum(axes[:, None, 1], anchors[None, :, 1])
        union = axes[:, None, 0] * axes[:, None, 1] + anchors[None, :, 0] * anchors[None, :, 1] - inter
        assign = (inter / np.maximum(union, 1e-9)).argmax(axis=1)
        new = anchors.copy()
        for bi in range(B):
            members = axes[assign == bi]
            if len(members):
                new[bi] = np.median(members, axis=0)
        if np.allclose(new, anchors):
            break
        anchors = new
    return anchors[np.argsort(anchors[:, 0] * anchors[:, 1])]


voc_root = find_voc_root(VOC_SEARCH_ROOT)
train_ds = VOCEllipseTorchDataset(voc_root, "train", MAX_TRAIN,
                                  random_rotation=TRAIN_RANDOM_ROTATION)
val_ds = VOCEllipseTorchDataset(voc_root, VAL_SPLIT, MAX_VAL,
                                random_rotation=False)
ANCHORS = estimate_anchors_from_samples(train_ds.samples, B)
ANCHORS_T = torch.tensor(ANCHORS, dtype=torch.float32, device=DEVICE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn)

print("VOC root:", voc_root)
print("train samples:", len(train_ds), "val samples:", len(val_ds))
print("anchors:", np.round(ANCHORS, 4).tolist())


In [ ]:
def denormalize_img(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = tensor.detach().cpu() * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()


def visualize_gt(dataset, n=6, save="figures/pretrained_gt_ellipses.png"):
    cols, rows = 3, math.ceil(n / 3)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, idx in zip(axes, range(min(n, len(dataset)))):
        img, targets = dataset[idx]
        ax.imshow(denormalize_img(img))
        for target in targets:
            ax.add_patch(MplEllipse(
                (target["cx"] * IMG_SIZE, target["cy"] * IMG_SIZE),
                2 * target["a"] * IMG_SIZE, 2 * target["b"] * IMG_SIZE,
                angle=np.rad2deg(target["theta"]), fill=False,
                edgecolor="lime", linestyle="--", lw=1.4))
        ax.set_title(f"sample {idx}", fontsize=9)
    fig.tight_layout()
    fig.savefig(save, dpi=120)
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=save))
    except Exception:
        plt.show()
    print(f"Saved → {save}")

visualize_gt(train_ds, n=6)


## Section 3 — Pretrained ResNet EllipseNet


In [ ]:
class PretrainedEllipseNet(nn.Module):
    """ResNet-18 backbone plus a YOLO-style ellipse prediction head."""
    def __init__(self, S=7, B=2, C=20, pretrained=True):
        super().__init__()
        if pretrained and TORCHVISION_WEIGHTS is not None:
            base = resnet18(weights=TORCHVISION_WEIGHTS)
        else:
            base = resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-2])  # N,512,H/32,W/32
        self.grid_pool = nn.AdaptiveAvgPool2d((S, S))       # 448 input -> 14x14 -> 7x7
        self.head = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(256, B * (6 + C), kernel_size=1),
        )
        self.S, self.B, self.C = S, B, C

    def forward(self, x):
        feat = self.grid_pool(self.backbone(x))
        out = self.head(feat)                      # N, B*(6+C), S, S
        out = out.permute(0, 2, 3, 1).contiguous() # N, 7, 7, B*(6+C)
        return out.view(x.size(0), self.S, self.S, self.B, 6 + self.C)

    def freeze_backbone(self, freeze=True):
        for p in self.backbone.parameters():
            p.requires_grad = not freeze


model = PretrainedEllipseNet(S=S, B=B, C=C, pretrained=True).to(DEVICE)
x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    y = model(x)
print("input:", tuple(x.shape))
print("output:", tuple(y.shape))
print("parameters:", sum(p.numel() for p in model.parameters()))
print("trainable:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## Section 4 — Target Assignment and Loss


In [ ]:
def build_targets_torch(targets_batch, anchors_t, S, B, C, device):
    N = len(targets_batch)
    obj = torch.zeros((N, S, S, B), dtype=torch.bool, device=device)
    t_cx = torch.zeros((N, S, S, B), device=device)
    t_cy = torch.zeros((N, S, S, B), device=device)
    t_a = torch.zeros((N, S, S, B), device=device)
    t_b = torch.zeros((N, S, S, B), device=device)
    t_th = torch.zeros((N, S, S, B), device=device)
    t_cls = torch.zeros((N, S, S, B), dtype=torch.long, device=device)

    anchors_np = anchors_t.detach().cpu().numpy()
    for ni, gts in enumerate(targets_batch):
        for gt in gts:
            cx, cy = float(gt["cx"]), float(gt["cy"])
            a, b = float(gt["a"]), float(gt["b"])
            gi = min(int(cx * S), S - 1)
            gj = min(int(cy * S), S - 1)
            inter = np.minimum(a, anchors_np[:, 0]) * np.minimum(b, anchors_np[:, 1])
            union = a * b + anchors_np[:, 0] * anchors_np[:, 1] - inter
            bi = int(np.argmax(inter / np.maximum(union, 1e-9)))
            obj[ni, gj, gi, bi] = True
            t_cx[ni, gj, gi, bi] = cx * S - gi
            t_cy[ni, gj, gi, bi] = cy * S - gj
            t_a[ni, gj, gi, bi] = math.log(max(a / anchors_np[bi, 0], 1e-6))
            t_b[ni, gj, gi, bi] = math.log(max(b / anchors_np[bi, 1], 1e-6))
            t_th[ni, gj, gi, bi] = float(gt["theta"])
            t_cls[ni, gj, gi, bi] = int(gt["class_id"])
    return obj, t_cx, t_cy, t_a, t_b, t_th, t_cls


def ellipse_loss_torch(raw, targets_batch, anchors_t):
    obj, t_cx, t_cy, t_a, t_b, t_th, t_cls = build_targets_torch(
        targets_batch, anchors_t, S, B, C, raw.device)
    noobj = ~obj

    conf_logits = raw[..., 0]
    loss_obj = F.binary_cross_entropy_with_logits(
        conf_logits[obj], torch.ones_like(conf_logits[obj])) if obj.any() else raw.sum() * 0
    loss_noobj = F.binary_cross_entropy_with_logits(
        conf_logits[noobj], torch.zeros_like(conf_logits[noobj])) if noobj.any() else raw.sum() * 0

    if not obj.any():
        total = L_OBJ * loss_obj + L_NOOBJ * loss_noobj
        return total, {"total": float(total.detach()), "loc": 0.0, "obj": float(loss_obj.detach()),
                       "noobj": float(loss_noobj.detach()), "cls": 0.0, "theta": 0.0}

    cx = torch.sigmoid(raw[..., 1])
    cy = torch.sigmoid(raw[..., 2])
    loss_center = F.mse_loss(cx[obj], t_cx[obj]) + F.mse_loss(cy[obj], t_cy[obj])

    raw_a = raw[..., 3]
    raw_b = raw[..., 4]
    loss_log_axes = 0.5 * F.mse_loss(raw_a[obj], t_a[obj]) + 0.5 * F.mse_loss(raw_b[obj], t_b[obj])

    pred_a = torch.exp(raw_a.clamp(-6, 2)) * anchors_t[:, 0]
    pred_b = torch.exp(raw_b.clamp(-6, 2)) * anchors_t[:, 1]
    tgt_a = torch.exp(t_a) * anchors_t[:, 0]
    tgt_b = torch.exp(t_b) * anchors_t[:, 1]
    loss_sqrt_axes = 0.5 * F.mse_loss(torch.sqrt(pred_a[obj].clamp_min(1e-8)), torch.sqrt(tgt_a[obj].clamp_min(1e-8)))
    loss_sqrt_axes += 0.5 * F.mse_loss(torch.sqrt(pred_b[obj].clamp_min(1e-8)), torch.sqrt(tgt_b[obj].clamp_min(1e-8)))
    loss_loc = loss_center + loss_log_axes + loss_sqrt_axes

    theta = np.pi * torch.sigmoid(raw[..., 5])
    diff = torch.abs(theta[obj] - t_th[obj])
    angle_err = torch.minimum(diff, np.pi - diff)
    loss_theta = angle_err.mean()

    loss_cls = F.cross_entropy(raw[..., 6:][obj], t_cls[obj])

    total = (L_LOC * loss_loc + L_OBJ * loss_obj + L_NOOBJ * loss_noobj
             + L_CLS * loss_cls + L_THETA * loss_theta)
    info = {
        "total": float(total.detach()), "loc": float(loss_loc.detach()),
        "obj": float(loss_obj.detach()), "noobj": float(loss_noobj.detach()),
        "cls": float(loss_cls.detach()), "theta": float(loss_theta.detach()),
    }
    return total, info


## Section 5 — Training Loop


In [ ]:
def make_optimizer(model, backbone_lr, head_lr):
    return torch.optim.AdamW([
        {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": backbone_lr},
        {"params": model.head.parameters(), "lr": head_lr},
    ], weight_decay=WEIGHT_DECAY)


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    keys = ["total", "loc", "obj", "noobj", "cls", "theta"]
    acc = {k: 0.0 for k in keys}
    n = 0
    for imgs, targets in loader:
        imgs = imgs.to(DEVICE)
        with torch.set_grad_enabled(training):
            raw = model(imgs)
            loss, info = ellipse_loss_torch(raw, targets, ANCHORS_T)
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
        for k in keys:
            acc[k] += info[k]
        n += 1
    return {k: acc[k] / max(n, 1) for k in keys}


def train_pretrained(model, epochs=EPOCHS):
    history = []
    best_val = float("inf")
    print(f"{'Ep':>4} | {'Train':>8} | {'Val':>8} | {'loc':>7} {'obj':>7} {'noobj':>7} {'cls':>7} {'theta':>7} | {'s':>5}")
    print("─" * 82)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        freeze = epoch <= FREEZE_BACKBONE_EPOCHS
        model.freeze_backbone(freeze)
        optimizer = make_optimizer(model, LR_BACKBONE, LR_HEAD)

        train_info = run_epoch(model, train_loader, optimizer)
        with torch.no_grad():
            val_info = run_epoch(model, val_loader, optimizer=None)

        elapsed = time.time() - t0
        print(f"{epoch:4d} | {train_info['total']:8.4f} | {val_info['total']:8.4f} | "
              f"{train_info['loc']:7.4f} {train_info['obj']:7.4f} {train_info['noobj']:7.4f} "
              f"{train_info['cls']:7.4f} {train_info['theta']:7.4f} | {elapsed:5.1f}")

        row = {"epoch": epoch, "train": train_info, "val": val_info,
               "backbone_frozen": freeze}
        history.append(row)

        if val_info["total"] < best_val:
            best_val = val_info["total"]
            torch.save({
                "model_state": model.state_dict(),
                "anchors": ANCHORS.tolist(),
                "config": {"S": S, "B": B, "C": C, "IMG_SIZE": IMG_SIZE},
            }, os.path.join(CHECKPOINT_DIR, "best.pt"))
            print(f"Checkpoint saved → {CHECKPOINT_DIR}/best.pt")

    with open(LOG_FILE, "w") as f:
        json.dump(history, f, indent=2)
    print(f"\nBest val loss: {best_val:.4f}   Log: {LOG_FILE}")
    return history


history = train_pretrained(model, EPOCHS)


## Section 6 — Training Curves


In [ ]:
def plot_history_torch(history, save="figures/pretrained_loss_curves.png"):
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, [row["train"]["total"] for row in history], marker="o", label="train")
    axes[0].plot(epochs, [row["val"]["total"] for row in history], marker="o", label="val")
    axes[0].set(title="Total loss", xlabel="epoch", ylabel="loss")
    axes[0].grid(alpha=0.3); axes[0].legend()

    for key in ["loc", "obj", "noobj", "cls", "theta"]:
        axes[1].plot(epochs, [row["train"][key] for row in history], marker="o", label=key)
    axes[1].set(title="Train loss components", xlabel="epoch", ylabel="loss")
    axes[1].grid(alpha=0.3); axes[1].legend()
    fig.tight_layout()
    fig.savefig(save, dpi=120)
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=save))
    except Exception:
        plt.show()
    print(f"Saved → {save}")

plot_history_torch(history)


## Section 7 — Decode, NMS, and Evaluation


In [ ]:
def ellipse_aabb_np(cx, cy, a, b, theta):
    tx = np.sqrt(a*a*np.cos(theta)**2 + b*b*np.sin(theta)**2)
    ty = np.sqrt(a*a*np.sin(theta)**2 + b*b*np.cos(theta)**2)
    return cx - tx, cy - ty, cx + tx, cy + ty


def aabb_iou_np(e1, e2):
    x1, y1, x2, y2 = ellipse_aabb_np(*e1)
    x3, y3, x4, y4 = ellipse_aabb_np(*e2)
    ix1, iy1 = max(x1, x3), max(y1, y3)
    ix2, iy2 = min(x2, x4), min(y2, y4)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area1 = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area2 = max(0.0, x4 - x3) * max(0.0, y4 - y3)
    return inter / max(area1 + area2 - inter, 1e-9)


def decode_single_torch(raw, anchors, conf_thresh=DEFAULT_CONF_THRESH):
    raw = raw.detach().cpu()
    grid = torch.arange(S)
    gy, gx = torch.meshgrid(grid, grid, indexing="ij")
    conf = torch.sigmoid(raw[..., 0])
    cx = (torch.sigmoid(raw[..., 1]) + gx[:, :, None]) / S
    cy = (torch.sigmoid(raw[..., 2]) + gy[:, :, None]) / S
    a = torch.exp(raw[..., 3].clamp(-6, 2)) * torch.tensor(anchors[:, 0])
    b = torch.exp(raw[..., 4].clamp(-6, 2)) * torch.tensor(anchors[:, 1])
    theta = np.pi * torch.sigmoid(raw[..., 5])
    cls_p = torch.softmax(raw[..., 6:], dim=-1)
    cls_score, cls_id = cls_p.max(dim=-1)
    score = conf * cls_score
    dets = []
    for gj, gi, bi in torch.nonzero(score > conf_thresh, as_tuple=False):
        dets.append((float(cx[gj, gi, bi]), float(cy[gj, gi, bi]),
                     float(a[gj, gi, bi]), float(b[gj, gi, bi]),
                     float(theta[gj, gi, bi]), float(score[gj, gi, bi]),
                     int(cls_id[gj, gi, bi])))
    return dets


def nms_dets(dets, iou_thresh=NMS_IOU_THRESH):
    dets = sorted(dets, key=lambda d: -d[5])
    keep = []
    while dets:
        best = dets.pop(0)
        keep.append(best)
        dets = [d for d in dets if aabb_iou_np(best[:5], d[:5]) < iou_thresh]
    return keep


def evaluate_pretrained(model, dataset, anchors, conf_thresh=DEFAULT_CONF_THRESH,
                        iou_thresh=EVAL_IOU_THRESH):
    model.eval()
    preds_by_cls = {c: [] for c in range(C)}
    n_gt_by_cls = {c: 0 for c in range(C)}
    with torch.no_grad():
        for idx in range(len(dataset)):
            img, gts = dataset[idx]
            for gt in gts:
                n_gt_by_cls[gt["class_id"]] += 1
            raw = model(img[None].to(DEVICE))[0]
            dets = nms_dets(decode_single_torch(raw, anchors, conf_thresh))
            matched = [False] * len(gts)
            for det in sorted(dets, key=lambda d: -d[5]):
                best_iou, best_j = 0.0, -1
                for j, gt in enumerate(gts):
                    if matched[j] or gt["class_id"] != det[6]:
                        continue
                    iou = aabb_iou_np(det[:5], (gt["cx"], gt["cy"], gt["a"], gt["b"], gt["theta"]))
                    if iou > best_iou:
                        best_iou, best_j = iou, j
                tp = best_iou >= iou_thresh and best_j >= 0
                if tp:
                    matched[best_j] = True
                preds_by_cls[det[6]].append((det[5], float(tp)))

    aps = {}
    for c in range(C):
        if n_gt_by_cls[c] == 0:
            continue
        entries = sorted(preds_by_cls[c], key=lambda x: -x[0])
        if not entries:
            aps[VOC_CLASSES[c]] = 0.0
            continue
        tp = np.cumsum([e[1] for e in entries])
        fp = np.cumsum([1 - e[1] for e in entries])
        prec = tp / np.maximum(tp + fp, 1e-8)
        rec = tp / max(n_gt_by_cls[c], 1)
        ap = 0.0
        for t in np.linspace(0, 1, 11):
            ap += (prec[rec >= t].max() if (rec >= t).any() else 0.0) / 11.0
        aps[VOC_CLASSES[c]] = float(ap)
    all_preds = [(s, tp) for c in range(C) for s, tp in preds_by_cls[c]]
    tp_total = sum(tp for _, tp in all_preds)
    return {
        "mAP": float(np.mean(list(aps.values()))) if aps else 0.0,
        "precision": float(tp_total / max(len(all_preds), 1)),
        "recall": float(tp_total / max(sum(n_gt_by_cls.values()), 1)),
        "per_class_AP": aps,
    }


def threshold_sweep_pretrained(model, dataset, anchors, thresholds=CONF_THRESHOLDS):
    print(f"{'conf':>6} {'mAP':>8} {'prec':>8} {'recall':>8}")
    rows = []
    for conf in thresholds:
        m = evaluate_pretrained(model, dataset, anchors, conf_thresh=conf)
        rows.append((conf, m))
        print(f"{conf:6.2f} {m['mAP']:8.4f} {m['precision']:8.4f} {m['recall']:8.4f}")
    return rows

ckpt = os.path.join(CHECKPOINT_DIR, "best.pt")
if os.path.exists(ckpt):
    # PyTorch 2.6+ defaults torch.load to weights_only=True, which rejects
    # older local checkpoints that stored NumPy arrays. This checkpoint is
    # produced by this notebook, so weights_only=False is appropriate here.
    payload = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    ckpt_cfg = payload.get("config", {})
    compatible = (ckpt_cfg.get("IMG_SIZE", IMG_SIZE) == IMG_SIZE
                  and ckpt_cfg.get("S", S) == S
                  and ckpt_cfg.get("B", B) == B
                  and ckpt_cfg.get("C", C) == C)
    if compatible:
        model.load_state_dict(payload["model_state"])
        if "anchors" in payload:
            ANCHORS = np.array(payload["anchors"], dtype=np.float32)
            ANCHORS_T = torch.tensor(ANCHORS, dtype=torch.float32, device=DEVICE)
        print("Checkpoint loaded →", ckpt)
    else:
        print(f"Skipping incompatible checkpoint: {ckpt}")
        print(f"  checkpoint config={ckpt_cfg}, current IMG_SIZE={IMG_SIZE}, S={S}, B={B}, C={C}")

print("Validation threshold sweep:")
threshold_rows = threshold_sweep_pretrained(model, val_ds, ANCHORS)


## Section 8 — Prediction Visualization


In [ ]:
def visualize_predictions_pretrained(model, dataset, anchors, n=6,
                                     conf_thresh=DEFAULT_CONF_THRESH,
                                     save="figures/pretrained_predictions.png"):
    model.eval()
    cols, rows = 3, math.ceil(n / 3)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")

    with torch.no_grad():
        for ax, idx in zip(axes, range(min(n, len(dataset)))):
            img, gts = dataset[idx]
            ax.imshow(denormalize_img(img))
            for gt in gts:
                ax.add_patch(MplEllipse(
                    (gt["cx"] * IMG_SIZE, gt["cy"] * IMG_SIZE),
                    2 * gt["a"] * IMG_SIZE, 2 * gt["b"] * IMG_SIZE,
                    angle=np.rad2deg(gt["theta"]), fill=False,
                    edgecolor="lime", linestyle="--", lw=1.2))
            raw = model(img[None].to(DEVICE))[0]
            dets = nms_dets(decode_single_torch(raw, anchors, conf_thresh))
            for det in dets:
                cx, cy, a, b, theta, score, cls_id = det
                ax.add_patch(MplEllipse(
                    (cx * IMG_SIZE, cy * IMG_SIZE), 2 * a * IMG_SIZE, 2 * b * IMG_SIZE,
                    angle=np.rad2deg(theta), fill=False, edgecolor="red", lw=1.5))
                ax.text(cx * IMG_SIZE, cy * IMG_SIZE, f"{VOC_CLASSES[cls_id]} {score:.2f}",
                        color="white", fontsize=6, ha="center", va="center",
                        bbox=dict(boxstyle="round,pad=0.1", fc="red", alpha=0.55))
            ax.set_title(f"sample {idx}", fontsize=9)

    fig.suptitle("Lime dashed = GT | Red solid = pretrained EllipseNet prediction", fontsize=10)
    fig.tight_layout()
    fig.savefig(save, dpi=120)
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=save))
    except Exception:
        plt.show()
    print(f"Saved → {save}")

visualize_predictions_pretrained(model, val_ds, ANCHORS, n=6,
                                 conf_thresh=DEFAULT_CONF_THRESH)


## Section 9 — Optional Pretrained Hyperparameter Search

Run small staged experiments that compare learning rates, loss coefficients, freeze duration, batch size, and confidence thresholds. This replaces the overfit check because the pretrained backbone should be evaluated by validation behavior and prediction quality.


In [ ]:
RUN_PRETRAINED_HPARAM_SEARCH = False

HPARAM_MAX_TRAIN = 200
HPARAM_MAX_VAL = 100
HPARAM_EPOCHS = 5
HPARAM_THRESHOLDS = [0.3, 0.5, 0.7]
HPARAM_ROOT = "checkpoints_pretrained_hparam"
HPARAM_LOG = "outputs/pretrained_hparam_summary.json"

HPARAM_CONFIGS = [
    dict(name="baseline",        lr_head=1e-3, lr_backbone=1e-5, batch_size=4, freeze_epochs=2,
         l_loc=5.0, l_obj=1.0, l_noobj=0.5, l_cls=1.0, l_theta=1.0),
    dict(name="lower_head_lr",   lr_head=5e-4, lr_backbone=1e-5, batch_size=4, freeze_epochs=2,
         l_loc=5.0, l_obj=1.0, l_noobj=0.5, l_cls=1.0, l_theta=1.0),
    dict(name="higher_head_lr",  lr_head=2e-3, lr_backbone=1e-5, batch_size=4, freeze_epochs=2,
         l_loc=5.0, l_obj=1.0, l_noobj=0.5, l_cls=1.0, l_theta=1.0),
    dict(name="more_localize",   lr_head=1e-3, lr_backbone=1e-5, batch_size=4, freeze_epochs=2,
         l_loc=7.5, l_obj=1.0, l_noobj=0.5, l_cls=1.0, l_theta=1.0),
    dict(name="more_objectness", lr_head=1e-3, lr_backbone=1e-5, batch_size=4, freeze_epochs=2,
         l_loc=5.0, l_obj=2.0, l_noobj=0.25, l_cls=1.0, l_theta=1.0),
    dict(name="short_freeze",    lr_head=1e-3, lr_backbone=1e-5, batch_size=4, freeze_epochs=1,
         l_loc=5.0, l_obj=1.0, l_noobj=0.5, l_cls=1.0, l_theta=1.0),
]


def safe_experiment_name(name):
    return name.replace("/", "_").replace(" ", "_")


def set_training_hparams(config):
    """Update globals used by train_pretrained and ellipse_loss_torch."""
    global LR_HEAD, LR_BACKBONE, BATCH_SIZE, FREEZE_BACKBONE_EPOCHS
    global L_LOC, L_OBJ, L_NOOBJ, L_CLS, L_THETA
    LR_HEAD = config["lr_head"]
    LR_BACKBONE = config["lr_backbone"]
    BATCH_SIZE = config["batch_size"]
    FREEZE_BACKBONE_EPOCHS = config["freeze_epochs"]
    L_LOC = config["l_loc"]
    L_OBJ = config["l_obj"]
    L_NOOBJ = config["l_noobj"]
    L_CLS = config["l_cls"]
    L_THETA = config["l_theta"]


def print_hparam_table(rows):
    if not rows:
        print("No hyperparameter rows to show.")
        return
    header = (f"{'config':<18} {'val':>8} {'conf':>6} {'mAP':>8} "
              f"{'prec':>8} {'recall':>8} {'lr_head':>9} {'loc':>5} {'obj':>5}")
    print(header)
    print("─" * len(header))
    for row in rows:
        print(f"{row['config']:<18} {row['best_val_loss']:8.4f} {row['conf_thresh']:6.2f} "
              f"{row['mAP']:8.4f} {row['precision']:8.4f} {row['recall']:8.4f} "
              f"{row['lr_head']:9.1e} {row['l_loc']:5.1f} {row['l_obj']:5.1f}")


def plot_pretrained_hparam_summary(rows, save="figures/pretrained_hparam_summary.png"):
    if not rows:
        return
    best_rows = []
    for name in sorted(set(row["config"] for row in rows)):
        sub = [row for row in rows if row["config"] == name]
        best_rows.append(max(sub, key=lambda row: row["mAP"]))
    labels = [row["config"] for row in best_rows]
    x = np.arange(len(labels))
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, key, title in zip(axes, ["mAP", "precision", "recall"], ["mAP", "Precision", "Recall"]):
        ax.bar(x, [row[key] for row in best_rows])
        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
        ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(save, dpi=120)
    try:
        from IPython.display import Image as IPyImage, display
        display(IPyImage(filename=save))
    except Exception:
        plt.show()
    print(f"Saved → {save}")


def run_pretrained_hparam_search(configs=HPARAM_CONFIGS):
    """Run staged pretrained experiments and evaluate each across thresholds."""
    global train_loader, val_loader, train_ds, val_ds, ANCHORS, ANCHORS_T
    global CHECKPOINT_DIR, LOG_FILE
    old_state = dict(
        train_loader=train_loader, val_loader=val_loader, train_ds=train_ds, val_ds=val_ds,
        ANCHORS=ANCHORS.copy(), ANCHORS_T=ANCHORS_T, CHECKPOINT_DIR=CHECKPOINT_DIR, LOG_FILE=LOG_FILE,
        LR_HEAD=LR_HEAD, LR_BACKBONE=LR_BACKBONE, BATCH_SIZE=BATCH_SIZE,
        FREEZE_BACKBONE_EPOCHS=FREEZE_BACKBONE_EPOCHS,
        L_LOC=L_LOC, L_OBJ=L_OBJ, L_NOOBJ=L_NOOBJ, L_CLS=L_CLS, L_THETA=L_THETA,
    )
    os.makedirs(HPARAM_ROOT, exist_ok=True)
    rows = []
    try:
        for config in configs:
            set_training_hparams(config)
            name = safe_experiment_name(config["name"])
            CHECKPOINT_DIR = os.path.join(HPARAM_ROOT, name)
            LOG_FILE = os.path.join(CHECKPOINT_DIR, "train_log.json")
            os.makedirs(CHECKPOINT_DIR, exist_ok=True)

            train_ds = VOCEllipseTorchDataset(voc_root, "train", HPARAM_MAX_TRAIN,
                                              random_rotation=TRAIN_RANDOM_ROTATION)
            val_ds = VOCEllipseTorchDataset(voc_root, "val", HPARAM_MAX_VAL,
                                            random_rotation=False)
            ANCHORS = estimate_anchors_from_samples(train_ds.samples, B)
            ANCHORS_T = torch.tensor(ANCHORS, dtype=torch.float32, device=DEVICE)
            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                      collate_fn=collate_fn)
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                    collate_fn=collate_fn)

            print(f"\n=== pretrained hparam: {name} ===")
            print(config)
            print("anchors:", np.round(ANCHORS, 4).tolist())
            exp_model = PretrainedEllipseNet(S=S, B=B, C=C, pretrained=True).to(DEVICE)
            exp_history = train_pretrained(exp_model, epochs=HPARAM_EPOCHS)
            best_val = min(row["val"]["total"] for row in exp_history)

            ckpt = os.path.join(CHECKPOINT_DIR, "best.pt")
            if os.path.exists(ckpt):
                payload = torch.load(ckpt, map_location=DEVICE, weights_only=False)
                exp_model.load_state_dict(payload["model_state"])

            for conf in HPARAM_THRESHOLDS:
                m = evaluate_pretrained(exp_model, val_ds, ANCHORS, conf_thresh=conf)
                rows.append({
                    "config": name,
                    "best_val_loss": float(best_val),
                    "conf_thresh": float(conf),
                    "mAP": float(m["mAP"]),
                    "precision": float(m["precision"]),
                    "recall": float(m["recall"]),
                    **config,
                })

        with open(HPARAM_LOG, "w") as f:
            json.dump(rows, f, indent=2)
        print(f"\nHyperparameter summary saved → {HPARAM_LOG}")
        print_hparam_table(rows)
        plot_pretrained_hparam_summary(rows)
        return rows
    finally:
        train_loader = old_state["train_loader"]
        val_loader = old_state["val_loader"]
        train_ds = old_state["train_ds"]
        val_ds = old_state["val_ds"]
        ANCHORS = old_state["ANCHORS"]
        ANCHORS_T = old_state["ANCHORS_T"]
        CHECKPOINT_DIR = old_state["CHECKPOINT_DIR"]
        LOG_FILE = old_state["LOG_FILE"]
        restore_config = dict(
            lr_head=old_state["LR_HEAD"], lr_backbone=old_state["LR_BACKBONE"],
            batch_size=old_state["BATCH_SIZE"], freeze_epochs=old_state["FREEZE_BACKBONE_EPOCHS"],
            l_loc=old_state["L_LOC"], l_obj=old_state["L_OBJ"],
            l_noobj=old_state["L_NOOBJ"], l_cls=old_state["L_CLS"], l_theta=old_state["L_THETA"],
        )
        set_training_hparams(restore_config)


if RUN_PRETRAINED_HPARAM_SEARCH:
    pretrained_hparam_rows = run_pretrained_hparam_search()
else:
    print("Pretrained hyperparameter search disabled. Set RUN_PRETRAINED_HPARAM_SEARCH=True to run it.")


## Section 10 — Final Summary


In [ ]:
print("=" * 58)
print("  EllipseNet Pretrained — Summary")
print("=" * 58)
print("  Backbone   : ResNet-18 pretrained on ImageNet")
print(f"  Dataset    : Pascal VOC 2012")
print(f"  Input size : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Grid       : {S}x{S} | B={B} | C={C}")
print(f"  Anchors    : {np.round(ANCHORS, 4).tolist()}")
print(f"  Epochs     : {EPOCHS} | Batch={BATCH_SIZE}")
print(f"  LR head    : {LR_HEAD} | LR backbone={LR_BACKBONE}")
print(f"  Checkpoint : {CHECKPOINT_DIR}/best.pt")
print("=" * 58)
